# Example 8.2.1 / 1: Check the Installation and the Model Size

Companion notebook for **8.2.1 Licenses and Memory Limits** in the AMPL lab Documentation, section "Check the Installation and the Model Size". It runs the page's example from top to bottom: the model, the amplpy way (data.py, run.py), then the pure-AMPL way (data.dat, model.run). Solver: HiGHS.

## Install Dependencies
💡 TIP: Visit [portal.ampl.com](https://portal.ampl.com/user/ampl/license/list) or [dev.ampl.com](https://dev.ampl.com/ampl/install.html) for AMPL setup on Colab, Docker, Python, Windows, Linux, macOS

In [ ]:
# Install dependencies
%pip install -q amplpy

# Google Colab & Kaggle integration
from amplpy import AMPL, ampl_notebook
ampl = ampl_notebook(
    modules=["highs"],      # modules to install
    license_uuid="default") # license to use

## Check the Installation and the Model Size

The first row is the easiest to reproduce. `python -m amplpy.modules list` at a terminal shows
the installed solver modules; a solver that is not among them cannot start. This example asks
for `mosekx`, a misspelled driver name, on the steel model. In `data.py` and `run.py`, `ampl` is
an `AMPL` object that has already read `model.mod`:

### model.mod
The model, written to a file that both ways of running it read.

In [2]:
%%writefile model.mod
set PROD;                                 # products
param rate {PROD} > 0;                    # tons produced per hour
param avail >= 0;                         # hours available
param profit {PROD};                      # profit per ton
param market {PROD} >= 0;                 # most tons that can be sold
var Make {p in PROD} >= 0, <= market[p];  # tons produced, up to the market limit
maximize Total_Profit: sum {p in PROD} profit[p] * Make[p];  # total profit
# production hours fit the hours available
subject to Time: sum {p in PROD} (1 / rate[p]) * Make[p] <= avail;

Writing model.mod


### amplpy: data.py and run.py

`ampl` is the `AMPL` object created above. Each run starts from a fresh session that has read `model.mod`; then `data.py` sets the data from Python and `run.py` solves and reads the results.

In [3]:
# A fresh session that has read model.mod, as data.py and run.py assume
ampl.reset()
ampl.eval("reset options;")  # reset keeps the options
ampl.read("model.mod")

In [4]:
# data.py
ampl.set["PROD"] = ["bands", "coils", "plate"]
ampl.param["rate"] = {"bands": 200, "coils": 140, "plate": 160}
ampl.param["profit"] = {"bands": 25, "coils": 30, "plate": 29}
ampl.param["market"] = {"bands": 6000, "coils": 4000, "plate": 3500}
ampl.param["avail"] = 40

In [5]:
# run.py
ampl.solve(solver="mosekx")
print(ampl.solve_result)

Cannot invoke mosekx: no such program.
failure


### Pure AMPL: data.dat and model.run

The same in pure AMPL, with the same `model.mod`:

In [6]:
%%writefile data.dat
set PROD := bands coils plate;
param rate := bands 200  coils 140  plate 160;
param profit := bands 25  coils 30  plate 29;
param market := bands 6000  coils 4000  plate 3500;
param avail := 40;

Writing data.dat


In [7]:
%%ampl_eval
reset;
reset options;
model model.mod;
data data.dat;

In [8]:
%%ampl_eval
# model.run
option solver mosekx;
solve;
display solve_result;

Cannot invoke mosekx: no such program.
solve_result = failure



### Another way in amplpy

`Cannot invoke ...: no such program` is an installation message: no solver ran, so nothing
about the model is known yet, and `solve_result` is `failure`. A licensing failure comes from a
solver that did start, with the vendor's licence message in the solve message instead.

A licence size limit or a memory shortage depends on the generated instance, so check its size:
`option show_stats 1` prints it at each solve, and `_nvars`, `_ncons` hold the counts. With the
same `model.mod` and `data.py`:

In [9]:
# A fresh session that has read model.mod, as data.py and run.py assume
ampl.reset()
ampl.eval("reset options;")  # reset keeps the options
ampl.read("model.mod")

In [10]:
# data.py
ampl.set["PROD"] = ["bands", "coils", "plate"]
ampl.param["rate"] = {"bands": 200, "coils": 140, "plate": 160}
ampl.param["profit"] = {"bands": 25, "coils": 30, "plate": 29}
ampl.param["market"] = {"bands": 6000, "coils": 4000, "plate": 3500}
ampl.param["avail"] = 40

In [11]:
# run.py
ampl.option["show_stats"] = 1
ampl.option["solver_msg"] = 0
ampl.solve(solver="highs")
print(ampl.get_value("_nvars"), "variables,", ampl.get_value("_ncons"), "constraints")


3 variables, all linear
1 constraint, all linear; 3 nonzeros
	1 inequality constraint
1 linear objective; 3 nonzeros.

3 variables, 1 constraints
